# Python Basics: Reproducible Data Cleaning with Pandas

This notebook introduces tools for data cleaning in pandas, the most popular Python library for cleaning, organizing, and analyzing data. 

## The "Excel Problem" of Data Cleaning

If you've done data cleaning before, you likely worked in a spreadsheet software like Excel or Google Sheets. While these platforms make it easy to view all your data at once and easily make changes to it, they introduce many opportunities for problems, such as human error in correcting data in individual cells or formatting issues with different data types.

But most importantly, **data cleaning in Excel isn't easily reproducible.** You don't get any record of the ways that you transformed your dataset (unless you write them down yourself), and if you need to clean a different dataset with the same structure, you will need to manually repeat all your steps. That typically won't be a problem for small datasets or cleaning workflows with only a few steps, but it quickly becomes tedious at best and error-prone at worst for more complicated scenarioes. 

This is where pandas comes in. **If you can write a script in Python for cleaning your data, then you can clean your data in the exact same way an infinite number of times.** Need to make a change to your workflow? Not a problem! You can just make the necessary change to your code and rerun it, knowing that all the other pieces of the process don't need to be changed.

## Importing Libraries

As we did last time, we start by importing the libraries that we will need to use in this notebook:

In [ ]:
import pandas as pd
import requests

## Review: Exploring Forecast Data

In our pandas introduction, we accessed the National Weather Service API to access upcoming forecast data, but that forecast has changed by now. Let's grab the data again with the same process and then explore it again:

In [ ]:
url = "https://api.weather.gov/gridpoints/LMK/94,71/forecast/hourly"
data = response_data = requests.get(url).json()
forecast = pd.DataFrame(data["properties"]["periods"])

Try writing your own code to answer the following questions about the `forecast` dataset (or just copy it from the previous lesson's notebook):
- How many rows and columns does it contain?
- What are the names of the columns?
- How many times does each forecast type appear in the `"shortForecast"` column?
- What values occur in the `"temperatureTrend"` column?
- For rows where `"isDayTime"` is True, how many times does each windspeed appear?

In [ ]:
# Explore the dataset here.
# You can write code for each answer in an individual cell,
# or you can wrap your code inside a print() statement
# so you can output all information in a single cell.


## Saving the Raw Data

Before we make any changes to our dataset, we should make sure that we save a raw copy of it. This copy will:
- Make sure we can restore the original version of the dataset if we make a mistake
- Serve as a point of comparison for our cleaned and transformed dataset

Recall that to save a DataFrame to a file, we can use the `.to_csv()` method of a DataFrame:

In [ ]:
# Save the raw forecast DataFrame to a csv:
forecast.to_csv("forecast_raw.csv", index=False)

## Reproducible Data Cleaning and Transformation in Pandas

We typically need to make some changes to datasets before we are able to use them. For our weather data, we will want to:
- Delete some extraneous columns
- Rename columns
- Create new columns
- Reformat data in some columns to be more useful
- Filter data by some condition

All or some of these may be tasks you're familiar with doing in Excel, and the point-and-click interface may feel more intuitive. But what if you were trying to make these changes to weather data for multiple locations? You'd have to redo them every single time on every single spreadsheet. This manual process would be very tedious, and you might make mistakes.

**By working in pandas, we can create a single data cleaning workflow, and then run it any number of times on data of the same structure.** Setting up the initial data cleaning process will take more work, but it will save time and prevent mistakes in the future.

We'll now go over how to clean our `forecast` DataFrame with the steps above, and will then put them together for a single workflow.

### Deleting Extraneous Columns with `.drop()`

We aren't interested in everything in our `forecast` DataFrame. To declutter it, we can remove columns using the `.drop()` method, in the format `forecast.drop("columnName", axis=1)`.

By default, `.drop()` will remove a **row**, not a **column**, from a DataFrame. For example, `forecast.drop(0)` would drop the first row (because its index is 0). Because we want to remove columns instead, we need add in `axis=1`. This tells Python to look for the specified name (or names) in the columns of the DataFrame, not the index.

The `"temperatureTrend"` column contains no information, so let's drop it from the DataFrame:

In [ ]:
forecast.drop("temperatureTrend", axis=1)

<h3 style="color:red; display:inline">Assign Your DataFrame to Its Variable Name as You Update It!</h3>

`forecast.drop("temperatureTrend", axis=1)` returned (and displayed) a new DataFrame without `"temperatureTrend"`, but that doesn't mean the column is gone from our `forecast` DataFrame.

Why not? **Because we didn't change the `forecast` variable.** If we want a change to our DataFrame (such as dropping a column) to be permanent, we need to assign the changed DataFrame to a variable:

In [ ]:
# Print the shape of the forecast DataFrame before we change it:
print(forecast.shape)

# Drop the temperatureTrend column, saving the changed DataFrame to the forecast variable:
forecast = forecast.drop("temperatureTrend", axis=1)

# Print the new shape of forecast:
print(forecast.shape)

Recall that the `.shape` attribute returns `(number of rows, number of columns)`, so we can see that our column was successfully dropped.

### The Test-Update-Repeat Workflow

A common workflow for data cleaning in pandas is to:
1. Write code to make a change to a DataFrame.
2. Examine the changed DataFrame to make sure it's been changed as intended.
3. Edit the code to update the variable for the DataFrame with the transformed version.

What if you make a mistake? Don't worry! Just remove your code that caused the mistake, rerun the code that loads your data to restore the original version (from a saved file), and then run the rest of your working code. That ability to unwind mistakes by rerunning a few lines of code is a key advantage of pandas over Excel.

### Renaming a Column with `.rename()`

All of the `"temperatureUnit"` values in the DataFrame are "F" (for Fahrenheit), so let's update the name of the `"temperature"` column to `"temperatureF"` and then drop the unit column.

To do so, we use `.rename()` in the form `forecast.rename({"currentName" : "newName"}, axis=1)`. Again, we need to specify `axis=1` so that pandas works with the columns of our DataFrame, not the index.

In [ ]:
forecast.rename({"temperature" : "temperatureF"}, axis=1)

Does the DataFrame look right to you? If so, save the updated version to the `forecast` variable. Then, repeat the process we used above to drop `"temperatureUnit"`

In [ ]:
# Rename the "temperature" column and save the changed version to forecast:
forecast = forecast.rename({"temperature" : "temperatureF"}, axis=1)
# Drop the "temperatureUnit" column. Once you have checked that the change
# was made as intended, save it to the forecast variable:


### Creating a New Column from an Existing One

Pandas makes it easy to transform all the data in a column at once. The simplest way to do this is with mathematical operators. 

For example, we could write code to return double the value of every temperature:

In [ ]:
forecast["temperatureF"] * 2

Remember: a single column in a DataFrame is a **Series**. When we perform a mathematical operation such as multiplication (`*`) on a Series, that operation gets performed on each value and returns a new Series containing those values.

How about converting the temperature in Fahrenheit to temperature in Celsius? The formula for doing so is `(5/9) * (temp_in_fahrenheit - 32)`. If we want to make this conversion for every value in `"temperatureF"`, we put those values in for `temp_in_fahrenheit`.

In [ ]:
(5/9) * (forecast["temperatureF"] - 32)

To save this data to a new column, we call the `forecast` DataFrame with the new column name we'd like to use in brackets:

In [ ]:
forecast["temperatureC"] = (5/9) * (forecast["temperatureF"] - 32)

In [ ]:
# Try writing code to view the "temperatureF" and "temperatureC" columns side by side:


## Reformatting Date and Time Data

Date and time information (or time-series data) is often an important part of any dataset, but it can be frustrating to work with. Even in pandas, the ins and outs of working with this type of data can be confusing. This section won't make you an expert, but it will demonstrate the flexibility and control that pandas can offer you if you work with time and date data frequently. 

In this dataset, you may have noticed that the `"startTime"` and `"endTime"` columns contain strange-looking values:

In [ ]:
forecast["startTime"]

These columns provide hyper-detailed date-time information, with units going from largest to smallest. For example, an entry such as `"2025-07-02T09:00:00-04:00"` contains the following information:
|Data|Meaning|
|---|---|
|`2025`|Year|
|`07`|Month|
|`02`|Day|
|`09`|Hour|
|`00`|Minutes|
|`00`|Seconds|
|`-04:00`|Timezone Offest (in this case, corresponding to EDT)|

This level of detail is not always helpful. For example, what if we wanted to group our forecast data by day rather than by hour? We would need dates (without additional time information) in their own column.

Pandas makes it possible to consistently convert date-time information into other formats, but the process takes several steps.

### Step 1: Convert a column with time information from a string object to a Timestamp object

Currently, the values contained in the `"startTime"` column are all string objects, meaning Python treats them as text data. We can confirm this by selecting one value and checking its type:

In [ ]:
sample_value = forecast["startTime"][0]
type(sample_value)

Pandas can convert these strings to a special data type, Timestamps, which can then be manipulated as date-time information. To do se, we use the `to_datetime()` function provided by pandas.

We'll assign these converted values to a variable `times`:

In [ ]:
# Convert the values in "startTime" from strings to Timestamps:
times = pd.to_datetime(forecast["startTime"])

# Check the type of the new values contained in the dt Series:
type(times[0])

In [ ]:
# Display the dt Series:
times

### Step 2: Reformat data to display in YYYY-MM-DD format

Now that we have converted our data to Timestamps, we can convert it once again to contain only the relevant information. In our case, we will preserve only the year, month, and day, but not the hour, minute, or timezone offset. 

To do se, we use the method `dt.strfime()`, passing in a string that indicates how we would like the resulting data to be displayed. We will use `"%Y-%m-%d"` to display first the year in four digits, the month in two digits, and the date in two digits:

In [ ]:
times.dt.strftime("%Y-%m-%d")

This `%` notation is very flexible and allows you to rewrite dates according to your needs, including reordering the information or adding in time in addition to date. Remembering all the date and time codes isn't intuitive, but you can find a complete list of them in the [Python documentation](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes).

If we wanted to, we could change the order of date information to the US convention of month-day-year:

In [ ]:
times.dt.strftime("%m-%d-%Y")

...or even add in the day of the week:

In [ ]:
times.dt.strftime("%a %m-%d-%Y")

Using the [Python documentation](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes), try experimenting with some other date-time information you can convey:

In [ ]:
# Try formatting your own date-time values here:


### Step 3: Create a new column for date values

Now that we know how to derive date information from our Timestamps, we can create a new column and call it `"Date"`, then populate it with our date values:

In [ ]:
forecast["date"] = times.dt.strftime("%Y-%m-%d")

We can now view the new `"date"` column in our `forecast` DataFrame:

In [ ]:
forecast

## Subsetting by Condition

A common data cleaning task is to reduce a dataset to only include values that meet a certain condition. We can do this via subsetting, which we discussed in the last lesson. Here, we can take advantage of our new date column to reduce our DataFrame down to only rows corresponding to today's date.

Recall that we create a **mask** by writing a **comparison operator** expression to check which rows in a column meet a given condition, and then pass that mask to our DataFrame:

In [ ]:
# Step 1: Create a mask for the condition we want to use to subset our data:
# NOTE: You will have to update the date here!
mask = forecast["date"] == "2025-07-02"

# Step 2: Pass the mask to the original DataFrame:
forecast[mask]

This time, let's take the additional step of assigning our subsetted DataFrame to its own variable:

In [ ]:
todays_forecast = forecast[mask]

## Saving the Transformed Dataset

We've now made a number of changes to the data we original accessed from the National Weather Service. We've dropped some columns we don't need, derived a new column for temperature in Celsius from an existing column, created a new column for the date, and finally, subset our DataFrame to only include today's weather. 

To round things off, let's save our transformed dataset to a new CSV:

In [ ]:
todays_forecast.to_csv("forecast_tpday.csv", index=False)

## Putting It All Together: Building a Reproducible Workflow

We now have a series of steps that will allow us to transform a dataset according to our specifications. Let's put it all together into a single code block.

Review the code above and copy the portions in the cell below that will:
1. Load the data from the National Weather Service API
2. Save the raw data to a CSV
3. Drop the `"temperatureTrend"` column
4. Rename the `"temperature"` column to `"temperatureF"`
5. Create the `"temperatureC"`column and drop the `"temperatureUnit"` column
6. Reformat the `"startTime"` column as a Timestamp and use it to create the `"date"` column
7. Save the final dataset to a new CSV

In [ ]:
# Bring together the full workflow here:


Congratulations! You now have a workflow that can be applied to any dataset in this format, whether that's future weather data for Lexington, KY, or data for another city. 

What's more, if you need to explain how you cleaned the data to a collaborator or need to remember what you did in the future, your code serves as documentation of your process, making it easy to know exactly what you did and make changes if necessary.

This is the power of code for reproducible data cleaning. It may take soem additonal time and energy to set up, but once it's been put together, it can be repeated as many times as needed, prevents errors due to manual data transformation, and makes any mistakes that do occur easier to spot and undo. 

# Continue Your Pandas Journey

This lesson has given you a taste of the power of pandas, but it's only scratched the surface of how you can automate and reproduce your data cleaning workflows. For the most thorough overview of everything pandas has to offer, see the e-book [Python for Data Analysis (3rd Edition)](https://wesmckinney.com/book/) by Wes McKinney, the creator of pandas.

For tutorial-based learning, download the following notebooks from Constellate and run them in Jupyter:
- **Pandas Basics 1:** https://github.com/ithaka/constellate-notebooks/blob/master/Pandas-basics/pandas-basics-1.ipynb
- **Pandas Basics 2:** https://github.com/ithaka/constellate-notebooks/blob/master/Pandas-basics/pandas-basics-2.ipynb
- **Pandas Basics 3:** https://github.com/ithaka/constellate-notebooks/blob/master/Pandas-basics/pandas-basics-3.ipynb
- **Pandas Intermediate 1:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-1.ipynb
- **Pandas Intermediate 2:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-2.ipynb
- **Pandas Intermediate 3:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-3.ipynb
- **Pandas Intermediate 4:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-4.ipynb
- **Pandas Intermediate 5:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-5.ipynb

# Acknowledgments
<img align="left" src="https://ithaka-labs.s3.amazonaws.com/static-files/images/tdm/tdmdocs/CC_BY.png"><br />

This lesson was created by Isaac Wink and is shared under a [Creative Commons CC BY License](https://creativecommons.org/licenses/by/4.0/).<br />